# 09 · Migrating from v1 to v2

The Review Radar story (00-08) shows what v2 looks like built fresh. This closing chapter
is for teams with a running v1 (`flytekit`/`union` SDK) estate: the concept map, a real
pipeline ported side-by-side, the behavioral differences that actually bite, and a
rollout strategy that de-risks the cutover.

**Learning goals**

1. Translate every v1 construct to its v2 counterpart
2. Re-author a representative v1 pipeline (workflow + map + actors) into v2
3. Know the semantic differences (no `@workflow` DSL, async, driver-task sizing)
4. Plan an incremental rollout

## 1. The concept map

| v1 (`flytekit` / `union`) | v2 (`flyte`) | Notes |
|---|---|---|
| `from flytekit import task, workflow` / `from union import ...` | `import flyte` | One package |
| `@task` | `@env.task` | Tasks belong to a `TaskEnvironment` |
| `@workflow` (compile-time DSL) | **gone** — any task orchestrates with plain Python | `if`/`for`/`try` just work |
| `@dynamic` | **gone** — same reason | |
| `conditional()` | native `if/elif/else` | |
| `map_task(fn)` / `union.map` | `flyte.map` / `flyte.map.aio` | `bound_inputs` → `functools.partial` |
| `ImageSpec(packages=[...])` | `flyte.Image.from_debian_base().with_pip_packages(...)` | Content-addressed builds |
| `Resources(mem="4Gi")` | `flyte.Resources(memory="4Gi")` | field renamed; tuples = (request, limit) |
| `cache=True, cache_version="v1"` | `cache=Cache(behavior="override", version_override="v1")` or `cache="auto"` | [04](./04-caching-and-reproducibility.ipynb) |
| `ActorEnvironment` | `TaskEnvironment(reusable=ReusePolicy(...))` | [05](./05-reusable-containers.ipynb) §5 |
| `@actor_cache` | `@lru_cache` / `@alru_cache` on an initializer | plain Python |
| `current_context().secrets.get(g, k)` | `flyte.Secret(key=..., as_env_var=...)` + `os.environ` | [02](./02-data-flow.ipynb) §3 |
| `current_context()` | `flyte.ctx()` | |
| `FlyteFile` / `FlyteDirectory` | `flyte.io.File` / `flyte.io.Dir` | async open/stream API |
| `LaunchPlan` + `CronSchedule` | `flyte.Trigger` + `flyte.Cron` on the task | deployed from files ([scripts/triggers_deploy.py](./scripts/triggers_deploy.py)) |
| `with_overrides(...)` | `.override(...)` | |
| Decks | Reports (`flyte.report`) | |
| `pyflyte run` / `union run` | `flyte run` | |
| `FlyteRemote` | `flyte.remote` | |

## 2. A real pipeline, ported

The two files below implement the **same pipeline** — train a model, then fan out
predictions across warm workers with a per-worker model cache. The v1 file uses the
classic patterns (`@workflow`, `union.map` with `bound_inputs`, `ActorEnvironment`,
`@actor_cache`); the v2 file is its faithful re-authoring.

- v1: [`scripts/migration/v1_pipeline.py`](./scripts/migration/v1_pipeline.py) *(reference only — needs the v1 SDK)*
- v2: [`scripts/migration/v2_pipeline.py`](./scripts/migration/v2_pipeline.py) *(runnable: `python scripts/migration/v2_pipeline.py`)*

Walk through them side-by-side; every block is annotated with its counterpart. The
structural differences:

1. **Composition moved into a task.** `@workflow def wf(...)` became `@main_env.task
   async def pipeline(...)`. There is no compile step — the pipeline is just Python that
   happens to call tasks, so debugging, conditionals, and loops behave like normal code.
2. **`bound_inputs` became `functools.partial`.** Standard-library idiom instead of API.
3. **Actors became a `ReusePolicy`.** Same warm-worker semantics, plus autoscaling
   `(min, max)` and per-pod/pool TTLs. `@actor_cache` became `functools.lru_cache`.
4. **Files got an async API.** `FlyteFile.download()` → `await file.download(path)` /
   streaming `open()`.

In [ ]:
# Optional: run the v2 side now (same cluster, ~2 min)
!python scripts/migration/v2_pipeline.py

## 3. What actually bites during migration

- **Everything can be async.** v1 tasks were sync; v2's primary model is `async def`.
  Sync tasks still work, but fan-out orchestration wants async. Adopt it top-down:
  orchestrating tasks first, leaf tasks as needed.
- **The driver task is a real pod.** In v1, `@workflow` logic ran in the platform's
  compiler. In v2 your orchestrating task *executes* — a 50k-item fan-out needs the
  driver sized for the tracking overhead ([03](./03-processing-at-scale.ipynb) §2).
- **No implicit global state.** Module-level code runs in *every* pod that imports the
  file. Expensive setup belongs behind `lru_cache`/`alru_cache` initializers.
- **Materialized I/O.** Returning a big list/dict inlines it into the action; v1 habits
  like passing DataFrames through workflow boundaries should become `File`/`DataFrame`
  references ([02](./02-data-flow.ipynb) §1).
- **Cache keys differ.** `cache=True, cache_version="v1"` maps to `behavior="override"` —
  don't port production tasks to `cache="auto"` unless you want code edits to invalidate.
- **Launch plans / schedules** are now `flyte.Trigger`s deployed with the task from a file.
- **UI/API parity check:** confirm anything the team relies on day-to-day (per-item map
  visibility is *better* in v2; some v1 views like Decks are replaced by Reports).

## 4. Rollout strategy

v1 and v2 SDKs coexist happily against the same platform — the strategy is incremental:

1. **Isolate:** create a `migration` project (or a dedicated domain) so v2 runs never
   collide with production v1 executions, quotas, or dashboards.
2. **Port a leaf pipeline first** — one with clear inputs/outputs and no launch-plan
   entanglement. Use the concept map; keep task names stable for cache/lineage sanity.
3. **Parallel-run:** schedule the v2 twin alongside v1 on real inputs; diff outputs for a
   week. `cache="auto"` during the port; pin `override` versions at parity sign-off.
4. **Re-point consumers** (triggers, `flyte.remote.Task.get` references, apps) at the v2
   tasks; deactivate the v1 launch plans.
5. **Repeat by team/pipeline.** Keep the v1 SDK pinned in legacy repos — nothing forces a
   big bang.

**Cutover checklist per pipeline:** outputs diffed ✓ · cache behavior pinned ✓ ·
schedules migrated + old ones deactivated ✓ · alerts/dashboards see the new runs ✓ ·
runbook updated (new CLI: `flyte run`, `flyte get run`) ✓

> 💬 **Discuss:** which pipeline is the customer's migration pilot? Sketch its v2 shape
> together — how many `TaskEnvironment`s, which get a `ReusePolicy`, and what cache
> behavior each task should pin.

## Further reading

- Union docs → User guide → *Migrating from Flyte 1 / Union 1*
- [appendix A](./appendix/A-deployment-adaptation.md) — platform-side items referenced
  throughout this series